# NDgpu — tri-S_N on GPU: level-scheduled sweeps on the HP-MR core (Colab)

Benchmarks the triangular-mesh discrete-ordinates solver's **level-scheduled
sweep engine** (`engine="levels"`, numpy/cupy via `device=`) on the 2D HP-MR
microreactor, drums inserted, SCB differencing, with **mesh refinement as the
problem-size axis**:

| experiment | compares | fixed |
|---|---|---|
| **i. GPU vs CPU** | `device="gpu"` vs `"cpu"`, same engine, same iteration sequence | full-core S_N (SCB), DSA + CMFD |
| **ii. hybrid vs full S_N** | transport in the drums only (Krylov-coupled to bulk diffusion) vs transport everywhere | device |

Both experiments report **accuracy next to cost**: k_eff (and Δpcm against the
named reference) alongside outers, transport sweeps, and wall time. CPU and GPU
run the identical iteration sequence, so the speed-up ratio is
tolerance-independent; the hybrid is a *different (approximate) discretization*,
so its Δpcm vs full S_N is a real modelling error, quoted alongside its speed.

(Cross sections are illustrative placeholders, not predictive.)

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

## Method

* **Engine**: `engine="levels"` on both devices — per ordinate the upwind
  dependency graph on the tri lattice is a DAG (vacuum bc), cells sort into
  topological levels, and a sweep is ~O(mesh diameter) sequential steps, each
  one batched update over *all* ordinates at once (batched 3×3 corner-block
  solves for SCB). On GPU each level is a handful of CUDA kernels; parallelism
  per level grows with mesh size, so expect the speed-up to grow with `refine`.
* **Warm-up**: one small untimed GPU solve first (compiles kernels, allocates
  the memory pool).
* **Timing**: `solve_seconds` is honest wall time — every sweep ends in a
  device→host reduction, which synchronizes. Setup (level scheduling, host
  numpy, device-independent) is reported separately.
* **Accelerations** (all defaults): DSA within groups, CMFD outers on the full
  S_N; the hybrid uses the monolithic Krylov interface coupling (one fused
  drum sweep + one bulk diffusion backsolve per matvec).

In [ ]:
import time
import numpy as np

from ndgpu.benchmarks import build_hpmr2d, hpmr_transport_mask
from ndgpu.tri_sn import TriSNTransportSolver
from ndgpu.hybrid_tri_sn import HybridTriSNDiffusionSolver

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

# Problem-size axis. refine=10/12 are worthwhile on a GPU runtime; on CPU they
# take minutes. NDGPU_QUICK=1 shrinks everything for a smoke test.
QUICK = bool(os.environ.get("NDGPU_QUICK"))
REFINES = [2, 3] if QUICK else [4, 6, 8, 10]
TOL = dict(tol_k=5e-7, tol_source=5e-6, max_outer=200)
QUAD = dict(n_polar=2, n_azi=8)


def problem(refine):
    p = build_hpmr2d(refine=refine, drum_angle_deg=0.0, absorber="polar")
    mix = dict(mix_material=p.mix_material, mix_weight=p.mix_weight)
    mask = hpmr_transport_mask(p, "drum").reshape(p.grid.shape)
    return p, mix, mask


def run_full_sn(refine, device):
    p, mix, _ = problem(refine)
    t = time.perf_counter()
    s = TriSNTransportSolver(p.grid, p.materials, p.material_map,
                             active=p.active, bc="vacuum", scheme="scb",
                             engine="levels", device=device, **QUAD, **mix)
    setup = time.perf_counter() - t
    r = s.solve(**TOL)
    assert r.converged
    return dict(refine=refine, cells=int(p.active.sum()), k=r.k_eff,
                outers=r.outer_iterations, sweeps=r.n_sweeps,
                setup=setup, solve=r.solve_seconds,
                graphs=s.graphs_active)


def run_hybrid(refine, device):
    p, mix, mask = problem(refine)
    t = time.perf_counter()
    h = HybridTriSNDiffusionSolver(p.grid, p.materials, p.material_map,
                                   sn_mask=mask, active=p.active,
                                   mask_bc=p.mask_bc, engine="levels",
                                   device=device, **QUAD, **mix)
    setup = time.perf_counter() - t
    t = time.perf_counter()
    r = h.solve(**TOL)
    solve = time.perf_counter() - t
    assert r.converged
    return dict(refine=refine, cells=int(p.active.sum()),
                sn_cells=int(mask.sum()), k=r.k_eff,
                outers=r.outer_iterations, sweeps=h.sn._sweep_count,
                setup=setup, solve=solve,
                graphs=h.sn.graphs_active)


if HAVE_GPU:                                # warm-up: compile kernels
    run_full_sn(2, "gpu"), run_hybrid(2, "gpu")

## Experiment i — GPU vs CPU, full-core S_N

Same engine, same iteration counts on both devices; the k columns must agree to
solver roundoff (the update algebra is identical). Reference for Δpcm: the CPU
run at the same refinement.

In [ ]:
rows_i = []
for refine in REFINES:
    cpu = run_full_sn(refine, "cpu")
    row = dict(refine=refine, cells=cpu["cells"], k_cpu=cpu["k"],
               outers=cpu["outers"], sweeps=cpu["sweeps"],
               t_cpu=cpu["solve"], setup=cpu["setup"])
    if HAVE_GPU:
        gpu = run_full_sn(refine, "gpu")
        row.update(k_gpu=gpu["k"], t_gpu=gpu["solve"],
                   speedup=cpu["solve"] / gpu["solve"],
                   graphs=gpu["graphs"])
    rows_i.append(row)

hdr = (f"{'refine':>6} {'cells':>7} {'k_cpu':>10} {'outers':>6} {'sweeps':>6} "
       f"{'t_cpu[s]':>9}")
if HAVE_GPU:
    hdr += f" {'t_gpu[s]':>9} {'dpcm':>6} {'speedup':>8} {'cudagraph':>9}"
print(hdr)
for r in rows_i:
    line = (f"{r['refine']:>6d} {r['cells']:>7d} {r['k_cpu']:>10.6f} "
            f"{r['outers']:>6d} {r['sweeps']:>6d} {r['t_cpu']:>9.2f}")
    if HAVE_GPU:
        cg = {True: 'on', False: 'fallback'}.get(r.get('graphs'), '?')
        line += (f" {r['t_gpu']:>9.2f} {(r['k_gpu']-r['k_cpu'])*1e5:>6.2f} "
                 f"{r['speedup']:>8.2f} {cg:>9}")
    print(line)

In [ ]:
import matplotlib.pyplot as plt

C_CPU, C_GPU, C_SN, C_HYB = "#2a78d6", "#008300", "#2a78d6", "#008300"
INK, MUTED = "#0b0b0b", "#52514e"


def style(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel, color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.set_title(title, color=INK, loc="left", fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(colors=MUTED)


cells = [r["cells"] for r in rows_i]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
ax = axes[0]
ax.loglog(cells, [r["t_cpu"] for r in rows_i], "-o", color=C_CPU, lw=2, ms=7,
          label="CPU")
if HAVE_GPU:
    ax.loglog(cells, [r["t_gpu"] for r in rows_i], "-o", color=C_GPU, lw=2,
              ms=7, label="GPU")
ax.legend(frameon=False, labelcolor=INK)
style(ax, "active cells", "solve wall time [s]",
      "i. Full-core tri-S_N solve time vs problem size")
ax = axes[1]
if HAVE_GPU:
    ax.semilogx(cells, [r["speedup"] for r in rows_i], "-o", color=C_CPU,
                lw=2, ms=7)
    ax.axhline(1.0, color=MUTED, lw=1, ls=":")
    style(ax, "active cells", "CPU time / GPU time",
          "GPU speed-up (same iteration sequence)")
else:
    ax.text(0.5, 0.5, "no GPU in this runtime", ha="center", va="center",
            color=MUTED, transform=ax.transAxes)
    ax.set_axis_off()
plt.show()

## Experiment ii — hybrid S_N/diffusion vs full-core S_N

Transport confined to the control-drum cells (Krylov-coupled to bulk
diffusion) against transport everywhere, at each refinement, on each available
device. The hybrid is an *approximation*: Δpcm vs the full S_N at the same
refinement is its modelling error, shown next to the speed-up it buys. The
drum fraction column shows how little of the core actually runs transport.

In [ ]:
devices = ["cpu"] + (["gpu"] if HAVE_GPU else [])
rows_ii = {d: [] for d in devices}
for refine in REFINES:
    base = {r["refine"]: r for r in rows_i}
    for d in devices:
        hyb = run_hybrid(refine, d)
        sn_t = base[refine]["t_cpu"] if d == "cpu" else base[refine]["t_gpu"]
        sn_k = base[refine]["k_cpu" if d == "cpu" else "k_gpu"]
        rows_ii[d].append(dict(refine=refine, cells=hyb["cells"],
                               frac=hyb["sn_cells"] / hyb["cells"],
                               k_sn=sn_k, k_hyb=hyb["k"],
                               dpcm=(hyb["k"] - sn_k) * 1e5,
                               t_sn=sn_t, t_hyb=hyb["solve"],
                               speedup=sn_t / hyb["solve"]))

for d in devices:
    print(f"-- {d} --")
    print(f"{'refine':>6} {'cells':>7} {'drum%':>6} {'k_SN':>10} {'k_hyb':>10} "
          f"{'dpcm':>7} {'t_SN[s]':>8} {'t_hyb[s]':>9} {'speedup':>8}")
    for r in rows_ii[d]:
        print(f"{r['refine']:>6d} {r['cells']:>7d} {100*r['frac']:>6.1f} "
              f"{r['k_sn']:>10.6f} {r['k_hyb']:>10.6f} {r['dpcm']:>7.0f} "
              f"{r['t_sn']:>8.2f} {r['t_hyb']:>9.2f} {r['speedup']:>8.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
dev = "gpu" if HAVE_GPU else "cpu"
rows = rows_ii[dev]
cells = [r["cells"] for r in rows]
ax = axes[0]
ax.loglog(cells, [r["t_sn"] for r in rows], "-o", color=C_SN, lw=2, ms=7,
          label="full S_N")
ax.loglog(cells, [r["t_hyb"] for r in rows], "-o", color=C_HYB, lw=2, ms=7,
          label="hybrid")
ax.legend(frameon=False, labelcolor=INK)
style(ax, "active cells", "solve wall time [s]",
      f"ii. Hybrid vs full S_N on {dev.upper()}")
ax = axes[1]
ax.semilogx(cells, [r["speedup"] for r in rows], "-o", color=C_SN, lw=2, ms=7)
ax.axhline(1.0, color=MUTED, lw=1, ls=":")
style(ax, "active cells", "full-S_N time / hybrid time",
      "hybrid speed-up (buys Δpcm above)")
plt.show()

## Reading the results

* **i.** CPU and GPU k agree to roundoff at every size (identical algebra). The
  GPU curve should flatten relative to CPU as refinement grows: per level the
  batch is (all ordinates) × (cells on the level), so utilization — and the
  speed-up — grows with problem size, while at small sizes per-level kernel
  launch overhead dominates and CPU can win.
* **ii.** The hybrid runs transport in only the drum fraction of cells and
  couples it to a prefactorized bulk diffusion solve, so its advantage is
  largest when transport is expensive (large meshes, CPU). Its Δpcm vs full
  S_N is the price; on this 12-drum core the isotropic interface
  reconstruction is known to overshoot the drum-worth correction (see
  `hybrid_tri_sn_hpmr.py`), so treat the hybrid as a fast approximation, not
  a replacement for the full-S_N reference.
* The bulk-diffusion backsolves and the CMFD/DSA LUs stay on the host by
  design (small sparse systems); only the sweeps move to the device.